# EvoVariant-TR baseline judge demo

This CPU/local notebook exposes the frozen evidence path. It does not train, tune, or open a holdout.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
if not (ROOT / "README.md").exists() and Path("/content/EvoVariant/README.md").exists():
    ROOT = Path("/content/EvoVariant")
print("repository:", ROOT)
assert (ROOT / "README.md").is_file()

## Frozen Phase 14 receipt

In [ ]:
receipt = json.loads((ROOT / "artifacts/phase14/phase14_locked_evo2_20260922.json").read_text())
rows = [json.loads(line) for line in (ROOT / "research/runs/formal_cpu_20260922/phase14_locked_evo2/predictions_with_local_labels.jsonl").read_text().splitlines() if line.strip()]
assert receipt["locked_cohort"]["completed_rows"] == len(rows) == 946
assert {row["split"] for row in rows} == {"LOCKED_TEST"}
assert all(row["remote_labels_transported"] is False for row in rows)
print(json.dumps({"cohort": receipt["locked_cohort"], "metrics": receipt["metrics"]}, indent=2))

## Headline metrics and confusion counts

The table is generated from the frozen receipt and joined predictions. The notebook does not refit a threshold or calibration.

In [ ]:
labels = [int(row["label"]) for row in rows]
predictions = [int(row["prediction"]) for row in rows]
tp = sum(y == 1 and p == 1 for y, p in zip(labels, predictions, strict=True))
tn = sum(y == 0 and p == 0 for y, p in zip(labels, predictions, strict=True))
fp = sum(y == 0 and p == 1 for y, p in zip(labels, predictions, strict=True))
fn = sum(y == 1 and p == 0 for y, p in zip(labels, predictions, strict=True))
print("TP TN FP FN:", tp, tn, fp, fn)
print((ROOT / "research/reports/final_metrics_table.json").read_text())

## Figures, provenance, and limits

The existing Phase 17 gallery contains the registered baseline families. The final-polish additions are generated by scripts/generate_final_readme_figures.py and indexed in artifacts/audits/FINAL_FIGURE_INVENTORY.json. The result is a temporal research benchmark, not clinical validation or a treatment claim.

In [ ]:
inventory = json.loads((ROOT / "artifacts/audits/FINAL_FIGURE_INVENTORY.json").read_text())
print("inherited Phase 17 families:", inventory["inherited_rendered_family_count"])
print("new final figures:", inventory["new_final_figures"])
print("new adaptation figures:", inventory["new_adaptation_figures"])
print("holdout opened for this demo: False")